In [15]:
from sqlalchemy import Engine, MetaData, Table, cast, func, update
from sqlalchemy.sql.sqltypes import BigInteger, DOUBLE_PRECISION, Integer, SmallInteger, Numeric, Double

import os
from pathlib import Path
import pandas as pd


from sql_utils import drop_table, build_postgres_engine
from sql_utils import print_table_head

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )
    
chunck_size = 100_000 # Se estiver estourando mt ram, abaixe esse numero

In [16]:
from sqlalchemy import Column, MetaData, Table, insert, inspect, null, select, String, delete, text
from sqlalchemy.schema import CreateSchema
from sqlalchemy.sql.type_api import TypeEngine

from sql_utils import normalize_table_columns_by_factor

target_schema_name = "cluster_stat"

def copy_table_between_schemas(
    engine: Engine,
    source_schema_name: str,
    source_table_name: str,
    target_schema_name: str,
    target_table_name: str | None = None,
    column_name_mapping: dict[str, str] | None = None,
    extra_columns: dict[str, TypeEngine] | None = None,
) -> None:
    target_table_name = target_table_name or source_table_name
    column_name_mapping = column_name_mapping or {}
    extra_columns = extra_columns or {}

    inspector = inspect(engine)
    if inspector.has_table(target_table_name, schema=target_schema_name):
        raise ValueError(f"Target table already exists: {target_schema_name}.{target_table_name}")

    source_metadata = MetaData(schema=source_schema_name)
    source_table = Table(source_table_name, source_metadata, autoload_with=engine)

    target_metadata = MetaData(schema=target_schema_name)
    target_columns = [
        Column(
            column_name_mapping.get(column.name, column.name),
            column.type,
            primary_key=column.primary_key,
            nullable=column.nullable,
        )
        for column in source_table.columns
    ]
    target_columns.extend(
        Column(column_name, column_type, nullable=True)
        for column_name, column_type in extra_columns.items()
    )
    target_table = Table(target_table_name, target_metadata, *target_columns)

    source_columns = list(source_table.columns)
    selected_columns = [*source_columns, *[null() for _ in extra_columns]]
    target_column_names = [column.name for column in target_table.columns]

    copy_statement = insert(target_table).from_select(
        target_column_names,
        select(*selected_columns),
    )

    with engine.begin() as connection:
        connection.execute(CreateSchema(target_schema_name, if_not_exists=True))
        target_table.create(connection)
        connection.execute(copy_statement)


def delete_missing_municipality_rows(
    engine: Engine,
    schema_name: str,
    table_name: str,
    territory_code_column: str,
    reference_table_name: str,
    reference_code_column: str,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)
    reference_table = Table(reference_table_name, metadata, autoload_with=engine)

    territory_code = cast(table.c[territory_code_column], String)
    reference_code = cast(reference_table.c[reference_code_column], String)

    statement = delete(table).where(
        func.length(territory_code) > 2,
        territory_code.not_in(select(reference_code)),
    )

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def delete_rows_below_threshold(
    engine: Engine,
    schema_name: str,
    table_name: str,
    column_name: str,
    threshold: int | float,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)

    statement = delete(table).where(table.c[column_name] < threshold)

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def rename_table_column(
    engine: Engine,
    schema_name: str,
    table_name: str,
    old_column_name: str,
    new_column_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    qualified_table_name = f"{preparer.quote_schema(schema_name)}.{preparer.quote(table_name)}"
    old_column = preparer.quote(old_column_name)
    new_column = preparer.quote(new_column_name)

    statement = text(
        f"ALTER TABLE {qualified_table_name} RENAME COLUMN {old_column} TO {new_column}"
    )

    with engine.begin() as connection:
        connection.execute(statement)

    print(f"Renamed column: {schema_name}.{table_name}.{old_column_name} -> {new_column_name}")

    


In [17]:
from typing import TypeAlias

from sqlalchemy.sql.elements import BinaryExpression, ColumnElement

ColumnRef: TypeAlias = tuple[str, str, str]
JoinPair: TypeAlias = tuple[ColumnRef, ColumnRef]
ColumnValueFilter: TypeAlias = tuple[ColumnRef, object]


def divide_columns_and_update(
    engine: Engine,
    numerator: ColumnRef,
    denominator: ColumnRef,
    target: ColumnRef,
    join_pairs: list[JoinPair] | None = None,
    round_result: bool = False,
    cast_result_to: TypeEngine | None = None,
    column_value_filters: list[ColumnValueFilter] | None = None,
) -> None:
    """
    Atualiza uma coluna com o resultado da divisão entre duas colunas.

    Exemplo:
        target = numerator / denominator

    As colunas podem estar:
    - na mesma tabela;
    - em tabelas diferentes;
    - e a coluna target pode ser uma das colunas usadas na operação.

    ColumnRef:
        (schema_name, table_name, column_name)

    JoinPair:
        ((schema_a, table_a, column_a), (schema_b, table_b, column_b))
    """

    join_pairs = join_pairs or []
    column_value_filters = column_value_filters or []

    metadata = MetaData()

    all_column_refs: list[ColumnRef] = [
        numerator,
        denominator,
        target,
    ]

    for left_ref, right_ref in join_pairs:
        all_column_refs.append(left_ref)
        all_column_refs.append(right_ref)

    for column_ref, _ in column_value_filters:
        all_column_refs.append(column_ref)

    table_refs = {
        (schema_name, table_name)
        for schema_name, table_name, _ in all_column_refs
    }

    tables: dict[tuple[str, str], Table] = {
        (schema_name, table_name): Table(
            table_name,
            metadata,
            schema=schema_name,
            autoload_with=engine,
        )
        for schema_name, table_name in table_refs
    }

    def get_column(column_ref: ColumnRef):
        schema_name, table_name, column_name = column_ref
        return tables[(schema_name, table_name)].c[column_name]

    numerator_col = get_column(numerator)
    denominator_col = get_column(denominator)
    target_col = get_column(target)

    target_schema, target_table_name, _ = target
    target_table = tables[(target_schema, target_table_name)]

    expression = numerator_col / func.nullif(denominator_col, 0)

    if round_result:
        expression = func.round(expression)

    if cast_result_to is not None:
        expression = cast(expression, cast_result_to)

    where_conditions: list[ColumnElement[bool]] = [
        cast(get_column(left_ref), BigInteger) == get_column(right_ref)
        for left_ref, right_ref in join_pairs
    ]

    where_conditions.extend(
        get_column(column_ref) == value
        for column_ref, value in column_value_filters
    )

    stmt = (
        update(target_table)
        .values({target_col.name: expression})
    )

    if where_conditions:
        stmt = stmt.where(*where_conditions)

    with engine.begin() as connection:
        connection.execute(stmt)

In [18]:
from sqlalchemy import text


def drop_schema(
    engine: Engine,
    schema_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    quoted_schema = preparer.quote_schema(schema_name)

    with engine.begin() as connection:
        connection.execute(text(f"DROP SCHEMA IF EXISTS {quoted_schema} CASCADE"))

    inspector = inspect(engine)
    inspector.clear_cache()
    if inspector.has_schema(schema_name):
        raise RuntimeError(f"Schema still exists after DROP SCHEMA CASCADE: {schema_name}")

    print(f"Dropped schema and all objects inside it: {schema_name}")


# Para usar quando quiser recomeçar:
drop_schema(engine, target_schema_name)

Dropped schema and all objects inside it: cluster_stat


In [19]:
print_table_head(engine, "norm_por_1k", "Pib_Geral", rows=100)

   id  Ano  Código da Grande Região Nome da Grande Região  Código da Unidade da Federação Sigla da Unidade da Federação Nome da Unidade da Federação     Cod                     Nome                  Região Metropolitana  Código da Mesorregião             Nome da Mesorregião  Código da Microrregião  Nome da Microrregião  Código da Região Geográfica Imediata Nome da Região Geográfica Imediata Município da Região Geográfica Imediata  Código da Região Geográfica Intermediária Nome da Região Geográfica Intermediária Município da Região Geográfica Intermediária  Código Concentração Urbana Nome Concentração Urbana   Tipo Concentração Urbana  Código Arranjo Populacional                                    Nome Arranjo Populacional     Hierarquia Urbana Hierarquia Urbana (principais categorias)  Código da Região Rural                                              Nome da Região Rural Região rural (segundo classificação do núcleo) Cidade-Região de São Paulo          PIB  PIB_per_capita            

In [20]:
target_table_name= "raw_count"

ext_columns={
    "pop_proj": DOUBLE_PRECISION(),
}

copy_table_between_schemas(
    engine, 
    "sinan_viol", 
    "contagens_mensais_violencia", 
    target_schema_name, 
    target_table_name, 
    extra_columns=ext_columns
)

In [22]:
metadata = MetaData()

raw_count = Table(
    "raw_count",
    metadata,
    schema=target_schema_name,
    autoload_with=engine,
)

pib_geral = Table(
    "Pib_Geral",
    metadata,
    schema="norm_por_1k",
    autoload_with=engine,
)

stmt = (
    update(raw_count)
    .values({"pop_proj": pib_geral.c.populacao_projetada})
    .where(
        cast(raw_count.c.ID_MN_OCOR, BigInteger) == cast(pib_geral.c.Cod, BigInteger),
        cast(func.extract("year", raw_count.c.mes), Integer) == cast(pib_geral.c.Ano, Integer),
    )
)

with engine.begin() as connection:
    result = connection.execute(stmt)

print(f"Updated rows: {result.rowcount}")

Updated rows: 138476


In [24]:
metadata = MetaData(schema=target_schema_name)

raw_count = Table(
    "raw_count",
    metadata,
    autoload_with=engine,
)

stmt = delete(raw_count).where(
    raw_count.c.pop_proj.is_(None)
)

with engine.begin() as connection:
    result = connection.execute(stmt)

print(f"Deleted rows: {result.rowcount}")

Deleted rows: 200938


In [25]:
print_table_head(engine, target_schema_name, target_table_name, rows=100)

       mes  ID_MN_OCOR SG_UF_OCOR  Total  CS_SEXO_f  CS_SEXO_i  CS_SEXO_m  LOCAL_OCOR_vazio  LOCAL_OCOR_bar_ou_similar  LOCAL_OCOR_comercio_servicos  LOCAL_OCOR_escola  LOCAL_OCOR_habitacao_coletiva  LOCAL_OCOR_ignorado  LOCAL_OCOR_industrias_construcao  LOCAL_OCOR_local_de_pratica_esportiva  LOCAL_OCOR_outro  LOCAL_OCOR_residencia  LOCAL_OCOR_via_publica  VIOL_FISIC_vazio  VIOL_FISIC_ignorado  VIOL_FISIC_nao  VIOL_FISIC_sim  VIOL_PSICO_vazio  VIOL_PSICO_ignorado  VIOL_PSICO_nao  VIOL_PSICO_sim  VIOL_TORT_vazio  VIOL_TORT_ignorado  VIOL_TORT_nao  VIOL_TORT_sim  VIOL_SEXU_vazio  VIOL_SEXU_ignorado  VIOL_SEXU_nao  VIOL_SEXU_sim  VIOL_TRAF_vazio  VIOL_TRAF_ignorado  VIOL_TRAF_nao  VIOL_TRAF_sim  VIOL_FINAN_vazio  VIOL_FINAN_ignorado  VIOL_FINAN_nao  VIOL_FINAN_sim  VIOL_NEGLI_vazio  VIOL_NEGLI_ignorado  VIOL_NEGLI_nao  VIOL_NEGLI_sim  VIOL_INFAN_vazio  VIOL_INFAN_ignorado  VIOL_INFAN_nao  VIOL_INFAN_sim  VIOL_LEGAL_vazio  VIOL_LEGAL_ignorado  VIOL_LEGAL_nao  VIOL_LEGAL_sim  VIOL_OUTR_vazi

Resumo das tabelas geradas/subidas

Tabela de perfis de renda

Esta tabela classifica cada município em um perfil latente de renda.

Como interpretar:

A coluna renda_perfil indica o perfil de renda mais provável do município.

A coluna renda_prob_max indica a confiança da classificação. Quanto mais perto de 1, mais seguro o modelo está de que o município pertence àquele perfil.

As colunas renda_prob_perfil_0, renda_prob_perfil_1, renda_prob_perfil_2 etc. mostram a probabilidade de o município pertencer a cada perfil de renda.

A coluna renda_entropia_bruta mede a incerteza da classificação. Quanto menor, mais concentrada está a probabilidade em um único perfil.

A coluna renda_incerteza_normalizada mede essa incerteza em escala de 0 a 1. Quanto menor, mais segura é a classificação.

A coluna renda_entropia_classificacao mede a qualidade da classificação em escala de 0 a 1. Quanto maior, mais clara é a separação do município naquele perfil.

Para interpretar os perfis de renda, deve-se olhar a média das faixas de renda dentro de cada renda_perfil.

Exemplo de interpretação:

Perfil com maior concentração nas faixas baixas: município de baixa renda.
Perfil com concentração em faixas intermediárias: município de renda intermediária.
Perfil com maior presença nas faixas altas: município de renda mais elevada.

Tabela de perfis de idade

Esta tabela classifica cada município em um perfil latente de estrutura etária.

Como interpretar:

A coluna idade_perfil indica o perfil etário mais provável do município.

A coluna idade_prob_max indica a confiança da classificação. Quanto mais perto de 1, mais seguro o modelo está.

As colunas idade_prob_perfil_0, idade_prob_perfil_1, idade_prob_perfil_2 etc. mostram a probabilidade de o município pertencer a cada perfil etário.

A coluna idade_entropia_bruta mede a incerteza da classificação. Quanto menor, mais segura é a classificação.

A coluna idade_incerteza_normalizada mede a incerteza em escala de 0 a 1. Quanto menor, menor a ambiguidade.

A coluna idade_entropia_classificacao mede a nitidez da classificação. Quanto maior, melhor definida é a classificação.

Para interpretar os perfis de idade, deve-se olhar a média das faixas etárias dentro de cada idade_perfil.

Exemplo de interpretação:

Perfil com alta proporção de jovens: município jovem.
Perfil com alta proporção de adultos: município adulto/intermediário.
Perfil com alta proporção de idosos: município envelhecido.

Tabela de perfis de instrucao

Esta tabela classifica cada município em um perfil latente de escolarização.

Como interpretar:

A coluna instrucao_perfil indica o perfil educacional mais provável do município.

A coluna instrucao_prob_max indica a confiança da classificação.

As colunas instrucao_prob_perfil_0, instrucao_prob_perfil_1, instrucao_prob_perfil_2 etc. mostram a probabilidade de o município pertencer a cada perfil de instrucao.

A coluna instrucao_entropia_bruta mede a incerteza da classificação. Quanto menor, mais segura é a classificação.

A coluna instrucao_incerteza_normalizada mede essa incerteza em escala de 0 a 1. Quanto menor, menor a ambiguidade.

A coluna instrucao_entropia_classificacao mede a qualidade da classificação em escala de 0 a 1. Quanto maior, mais nítida é a classificação.

Para interpretar os perfis de instrucao, deve-se olhar a média das faixas de instrucao dentro de cada instrucao_perfil.

Exemplo de interpretação:

Perfil com alta proporção de baixa instrucao: município de baixa escolarização.
Perfil com concentração em ensino médio: município de instrucao intermediária.
Perfil com maior proporção de ensino superior: município de instrucao elevada.

In [26]:
print_table_head(engine, "clusters", "dist_idade", rows=10)

 index       id cod_territorio                 territorio  0_4_homens  0_4_mulheres  100_mais_homens  100_mais_mulheres  10_14_homens  10_14_mulheres  15_19_homens  15_19_mulheres  20_24_homens  20_24_mulheres  25_29_homens  25_29_mulheres  30_34_homens  30_34_mulheres  35_39_homens  35_39_mulheres  40_44_homens  40_44_mulheres  45_49_homens  45_49_mulheres  50_54_homens  50_54_mulheres  55_59_homens  55_59_mulheres  5_9_homens  5_9_mulheres  60_64_homens  60_64_mulheres  65_69_homens  65_69_mulheres  70_74_homens  70_74_mulheres  75_79_homens  75_79_mulheres  80_84_homens  80_84_mulheres  85_89_homens  85_89_mulheres  90_94_homens  90_94_mulheres  95_99_homens  95_99_mulheres  idade_perfil  idade_prob_max  idade_entropia_bruta  idade_incerteza_normalizada  idade_entropia_classificacao  idade_prob_perfil_0  idade_prob_perfil_1  idade_prob_perfil_2  idade_prob_perfil_3  idade_prob_perfil_4  idade_prob_perfil_5
     0 11000151        1100015 Alta Floresta D'Oeste (RO)   73.404648      0.

In [27]:
print_table_head(engine, "clusters", "dist_instrucao", rows=10)

 index    id Brasil, Grande Região, Unidade da Federação e Município     Cod  Fundamental completo e médio incompleto_homens  Fundamental completo e médio incompleto_mulheres  Médio completo e superior incompleto_homens  Médio completo e superior incompleto_mulheres  Sem instrução e fundamental incompleto_homens  Sem instrução e fundamental incompleto_mulheres  Superior completo_homens  Superior completo_mulheres  instrucao_perfil  instrucao_prob_max  instrucao_entropia_bruta  instrucao_incerteza_normalizada  instrucao_entropia_classificacao  instrucao_prob_perfil_0  instrucao_prob_perfil_1  instrucao_prob_perfil_2  instrucao_prob_perfil_3  instrucao_prob_perfil_4  instrucao_prob_perfil_5
     0 10000                                            Campo Alegre 2701407                                      167.916667                                               0.0                                   345.208333                                            0.0                                    

In [28]:
print_table_head(engine, "clusters", "dist_renda", rows=10)

 index  id  Cod Brasil, Unidade da Federação e Município  Até 1/4 de salário mínimo_homens  Até 1/4 de salário mínimo_mulheres  Mais de 1 a 2 salários mínimos_homens  Mais de 1 a 2 salários mínimos_mulheres  Mais de 1/2 a 1 salário mínimo_homens  Mais de 1/2 a 1 salário mínimo_mulheres  Mais de 1/4 a 1/2 salário mínimo_homens  Mais de 1/4 a 1/2 salário mínimo_mulheres  Mais de 10 a 15 salários mínimos_homens  Mais de 10 a 15 salários mínimos_mulheres  Mais de 15 a 20 salários mínimos_homens  Mais de 15 a 20 salários mínimos_mulheres  Mais de 2 a 3 salários mínimos_homens  Mais de 2 a 3 salários mínimos_mulheres  Mais de 20 salários mínimos_homens  Mais de 20 salários mínimos_mulheres  Mais de 3 a 5 salários mínimos_homens  Mais de 3 a 5 salários mínimos_mulheres  Mais de 5 a 10 salários mínimos_homens  Mais de 5 a 10 salários mínimos_mulheres  Sem rendimento_homens  Sem rendimento_mulheres  renda_perfil  renda_prob_max  renda_entropia_bruta  renda_incerteza_normalizada  renda_entropia_

# Agregações e estatísticas por cluster

In [31]:
from sqlalchemy import literal

cluster_aggregation_configs = [
    {
        "source_table": "dist_idade",
        "code_column_candidates": ["cod_territorio", "cod_do_territorio", "Cod"],
        "profile_column": "idade_perfil",
        "probability_column_candidates": ["idade_prob_max"],
        "stats_table": "estatisticas_cluster_idade",
        "outlier_table": "outliers_cluster_idade",
    },
    {
        "source_table": "dist_instrucao",
        "code_column_candidates": ["Cod"],
        "profile_column": "instrucao_perfil",
        "probability_column_candidates": ["instrucao_prob_max"],
        "stats_table": "estatisticas_cluster_instrucao",
        "outlier_table": "outliers_cluster_instrucao",
    },
    {
        "source_table": "dist_renda",
        "code_column_candidates": ["Cod"],
        "profile_column": "renda_perfil",
        "probability_column_candidates": ["renda_prob_max"],
        "stats_table": "estatisticas_cluster_renda",
        "outlier_table": "outliers_cluster_renda",
    },
]

numeric_types = (
    Integer,
    BigInteger,
    SmallInteger,
    Numeric,
    Double,
    DOUBLE_PRECISION,
)

cluster_profile_columns = [
    config["profile_column"]
    for config in cluster_aggregation_configs
]

raw_count_identifier_columns = {
    "mes",
    "ID_MN_OCOR",
    "SG_UF_OCOR",
    "Ano",
    "pop_proj",
    *cluster_profile_columns,
}


def quote_qualified_table_name(engine: Engine, schema_name: str, table_name: str) -> str:
    preparer = engine.dialect.identifier_preparer
    return f"{preparer.quote_schema(schema_name)}.{preparer.quote(table_name)}"


def drop_table_if_exists(engine: Engine, schema_name: str, table_name: str) -> None:
    qualified_table_name = quote_qualified_table_name(engine, schema_name, table_name)

    with engine.begin() as connection:
        connection.execute(text(f"DROP TABLE IF EXISTS {qualified_table_name} CASCADE"))


def add_integer_column_if_missing(
    engine: Engine,
    schema_name: str,
    table_name: str,
    column_name: str,
) -> None:
    inspector = inspect(engine)
    existing_columns = {
        column["name"]
        for column in inspector.get_columns(table_name, schema=schema_name)
    }

    if column_name in existing_columns:
        return

    qualified_table_name = quote_qualified_table_name(engine, schema_name, table_name)
    column = engine.dialect.identifier_preparer.quote(column_name)

    with engine.begin() as connection:
        connection.execute(text(f"ALTER TABLE {qualified_table_name} ADD COLUMN {column} integer"))


def resolve_existing_column(table: Table, column_candidates: list[str]) -> str:
    existing_columns = set(table.c.keys())

    for column_name in column_candidates:
        if column_name in existing_columns:
            return column_name

    raise ValueError(
        f"None of the candidate columns {column_candidates} exist in {table.fullname}."
    )


def resolve_optional_column(table: Table, column_candidates: list[str]):
    existing_columns = set(table.c.keys())

    for column_name in column_candidates:
        if column_name in existing_columns:
            return table.c[column_name]

    return literal(1.0)


def get_raw_count_table() -> Table:
    metadata = MetaData()

    return Table(
        "raw_count",
        metadata,
        schema=target_schema_name,
        autoload_with=engine,
    )


def get_count_column_names(table: Table) -> list[str]:
    return [
        column.name
        for column in table.columns
        if column.name not in raw_count_identifier_columns
        and isinstance(column.type, numeric_types)
    ]


def print_column_summary(table: Table, count_column_names: list[str]) -> None:
    print(f"Count columns: {len(count_column_names)}")
    print("First columns:", count_column_names[:10])
    print("Cluster profile columns:", cluster_profile_columns)


In [32]:
for profile_column in cluster_profile_columns:
    add_integer_column_if_missing(
        engine,
        target_schema_name,
        "raw_count",
        profile_column,
    )

for config in cluster_aggregation_configs:
    metadata = MetaData()
    raw_count = Table(
        "raw_count",
        metadata,
        schema=target_schema_name,
        autoload_with=engine,
    )
    cluster_table = Table(
        config["source_table"],
        metadata,
        schema="clusters",
        autoload_with=engine,
    )

    code_column_name = resolve_existing_column(
        cluster_table,
        config["code_column_candidates"],
    )
    profile_column_name = config["profile_column"]

    municipality_code = cast(cluster_table.c[code_column_name], BigInteger).label("municipality_code")
    probability = cast(
        resolve_optional_column(cluster_table, config["probability_column_candidates"]),
        Double,
    )

    ranked_profiles = (
        select(
            municipality_code,
            cast(cluster_table.c[profile_column_name], Integer).label("profile"),
            func.row_number()
            .over(
                partition_by=municipality_code,
                order_by=probability.desc(),
            )
            .label("profile_rank"),
        )
        .where(cluster_table.c[profile_column_name].is_not(None))
        .subquery()
    )

    cluster_lookup = (
        select(
            ranked_profiles.c.municipality_code,
            ranked_profiles.c.profile,
        )
        .where(ranked_profiles.c.profile_rank == 1)
        .subquery()
    )

    stmt = (
        update(raw_count)
        .values({profile_column_name: cluster_lookup.c.profile})
        .where(
            cast(raw_count.c.ID_MN_OCOR, BigInteger) == cluster_lookup.c.municipality_code
        )
    )

    with engine.begin() as connection:
        result = connection.execute(stmt)

    print(f"{profile_column_name}: updated rows {result.rowcount}")

raw_count = get_raw_count_table()
count_column_names = get_count_column_names(raw_count)
print_column_summary(raw_count, count_column_names)


idade_perfil: updated rows 138476
instrucao_perfil: updated rows 136093
renda_perfil: updated rows 138476
Count columns: 413
First columns: ['Total', 'CS_SEXO_f', 'CS_SEXO_i', 'CS_SEXO_m', 'LOCAL_OCOR_vazio', 'LOCAL_OCOR_bar_ou_similar', 'LOCAL_OCOR_comercio_servicos', 'LOCAL_OCOR_escola', 'LOCAL_OCOR_habitacao_coletiva', 'LOCAL_OCOR_ignorado']
Cluster profile columns: ['idade_perfil', 'instrucao_perfil', 'renda_perfil']


In [33]:
raw_count = get_raw_count_table()
count_column_names = get_count_column_names(raw_count)

drop_table_if_exists(engine, target_schema_name, "raw_count_anual")

annual_metadata = MetaData(schema=target_schema_name)
raw_count_anual = Table(
    "raw_count_anual",
    annual_metadata,
    Column("Ano", Integer),
    Column("ID_MN_OCOR", raw_count.c.ID_MN_OCOR.type),
    Column("SG_UF_OCOR", raw_count.c.SG_UF_OCOR.type),
    *(Column(profile_column, Integer) for profile_column in cluster_profile_columns),
    Column("pop_proj", DOUBLE_PRECISION),
    *(Column(column_name, DOUBLE_PRECISION) for column_name in count_column_names),
)

year_expression = cast(func.extract("year", raw_count.c.mes), Integer)

annual_select = (
    select(
        year_expression.label("Ano"),
        raw_count.c.ID_MN_OCOR,
        raw_count.c.SG_UF_OCOR,
        *(raw_count.c[profile_column] for profile_column in cluster_profile_columns),
        func.max(cast(raw_count.c.pop_proj, Double)).label("pop_proj"),
        *(
            func.sum(
                func.coalesce(cast(raw_count.c[column_name], Double), 0)
            ).label(column_name)
            for column_name in count_column_names
        ),
    )
    .group_by(
        year_expression,
        raw_count.c.ID_MN_OCOR,
        raw_count.c.SG_UF_OCOR,
        *(raw_count.c[profile_column] for profile_column in cluster_profile_columns),
    )
)

with engine.begin() as connection:
    raw_count_anual.create(connection)
    result = connection.execute(
        insert(raw_count_anual).from_select(
            [column.name for column in raw_count_anual.columns],
            annual_select,
        )
    )

print(f"raw_count_anual rows: {result.rowcount}")


raw_count_anual rows: 15139


In [34]:
def municipal_rate(column_name: str, annual_table: Table):
    return (
        cast(annual_table.c[column_name], Double)
        / func.nullif(cast(annual_table.c.pop_proj, Double), 0)
        * 1000
    )


def cluster_rate(column_name: str, annual_table: Table):
    return (
        func.sum(func.coalesce(cast(annual_table.c[column_name], Double), 0))
        / func.nullif(func.sum(cast(annual_table.c.pop_proj, Double)), 0)
        * 1000
    )


def create_stats_table(config: dict[str, object]) -> None:
    metadata = MetaData()
    annual_table = Table(
        "raw_count_anual",
        metadata,
        schema=target_schema_name,
        autoload_with=engine,
    )
    count_column_names = get_count_column_names(annual_table)
    profile_column = config["profile_column"]
    stats_table_name = config["stats_table"]

    drop_table_if_exists(engine, target_schema_name, stats_table_name)

    target_metadata = MetaData(schema=target_schema_name)
    stats_table = Table(
        stats_table_name,
        target_metadata,
        Column("Ano", Integer),
        Column("Cod", Integer),
        Column("coluna_contagem", String),
        Column("total_coluna", DOUBLE_PRECISION),
        Column("pop_proj", DOUBLE_PRECISION),
        Column("total_coluna_por_mil", DOUBLE_PRECISION),
        Column("media", DOUBLE_PRECISION),
        Column("mediana", DOUBLE_PRECISION),
        Column("variancia", DOUBLE_PRECISION),
        Column("desvio", DOUBLE_PRECISION),
        Column("q1", DOUBLE_PRECISION),
        Column("q3", DOUBLE_PRECISION),
        Column("limite_outlier_superior", DOUBLE_PRECISION),
    )

    with engine.begin() as connection:
        stats_table.create(connection)

        for column_name in count_column_names:
            rate_expression = municipal_rate(column_name, annual_table)
            q1_expression = func.percentile_cont(0.25).within_group(rate_expression)
            q3_expression = func.percentile_cont(0.75).within_group(rate_expression)

            stats_select = (
                select(
                    annual_table.c.Ano.label("Ano"),
                    cast(annual_table.c[profile_column], Integer).label("Cod"),
                    literal(column_name).label("coluna_contagem"),
                    func.sum(
                        func.coalesce(cast(annual_table.c[column_name], Double), 0)
                    ).label("total_coluna"),
                    func.sum(cast(annual_table.c.pop_proj, Double)).label("pop_proj"),
                    cluster_rate(column_name, annual_table).label("total_coluna_por_mil"),
                    func.avg(rate_expression).label("media"),
                    func.percentile_cont(0.5).within_group(rate_expression).label("mediana"),
                    func.var_samp(rate_expression).label("variancia"),
                    func.stddev_samp(rate_expression).label("desvio"),
                    q1_expression.label("q1"),
                    q3_expression.label("q3"),
                    (q3_expression + 1.5 * (q3_expression - q1_expression)).label(
                        "limite_outlier_superior"
                    ),
                )
                .where(annual_table.c[profile_column].is_not(None))
                .group_by(annual_table.c.Ano, annual_table.c[profile_column])
                .order_by(annual_table.c.Ano, annual_table.c[profile_column])
            )

            connection.execute(
                insert(stats_table).from_select(
                    [column.name for column in stats_table.columns],
                    stats_select,
                )
            )

    print(f"{target_schema_name}.{stats_table_name}: inserted {len(count_column_names)} count columns")


def create_outlier_table(config: dict[str, object]) -> None:
    metadata = MetaData()
    annual_table = Table(
        "raw_count_anual",
        metadata,
        schema=target_schema_name,
        autoload_with=engine,
    )
    stats_table = Table(
        config["stats_table"],
        metadata,
        schema=target_schema_name,
        autoload_with=engine,
    )
    count_column_names = get_count_column_names(annual_table)
    profile_column = config["profile_column"]
    outlier_table_name = config["outlier_table"]

    drop_table_if_exists(engine, target_schema_name, outlier_table_name)

    target_metadata = MetaData(schema=target_schema_name)
    outlier_table = Table(
        outlier_table_name,
        target_metadata,
        Column("Ano", Integer),
        Column("Cod", Integer),
        Column("ID_MN_OCOR", annual_table.c.ID_MN_OCOR.type),
        Column("SG_UF_OCOR", annual_table.c.SG_UF_OCOR.type),
        Column("coluna_contagem", String),
        Column("valor_municipio", DOUBLE_PRECISION),
        Column("pop_proj", DOUBLE_PRECISION),
        Column("taxa_municipal_por_mil", DOUBLE_PRECISION),
        Column("limite_outlier_superior", DOUBLE_PRECISION),
    )

    with engine.begin() as connection:
        outlier_table.create(connection)

        for column_name in count_column_names:
            rate_expression = municipal_rate(column_name, annual_table)

            outlier_select = (
                select(
                    annual_table.c.Ano.label("Ano"),
                    cast(annual_table.c[profile_column], Integer).label("Cod"),
                    annual_table.c.ID_MN_OCOR,
                    annual_table.c.SG_UF_OCOR,
                    literal(column_name).label("coluna_contagem"),
                    cast(annual_table.c[column_name], Double).label("valor_municipio"),
                    cast(annual_table.c.pop_proj, Double).label("pop_proj"),
                    rate_expression.label("taxa_municipal_por_mil"),
                    stats_table.c.limite_outlier_superior,
                )
                .select_from(
                    annual_table.join(
                        stats_table,
                        (annual_table.c.Ano == stats_table.c.Ano)
                        & (cast(annual_table.c[profile_column], Integer) == stats_table.c.Cod)
                        & (stats_table.c.coluna_contagem == column_name),
                    )
                )
                .where(
                    annual_table.c[profile_column].is_not(None),
                    rate_expression > stats_table.c.limite_outlier_superior,
                )
                .order_by(
                    annual_table.c.Ano,
                    annual_table.c[profile_column],
                    annual_table.c.ID_MN_OCOR,
                )
            )

            connection.execute(
                insert(outlier_table).from_select(
                    [column.name for column in outlier_table.columns],
                    outlier_select,
                )
            )

    print(f"{target_schema_name}.{outlier_table_name}: inserted outlier rows")


for config in cluster_aggregation_configs:
    create_stats_table(config)
    create_outlier_table(config)


cluster_stat.estatisticas_cluster_idade: inserted 413 count columns
cluster_stat.outliers_cluster_idade: inserted outlier rows
cluster_stat.estatisticas_cluster_instrucao: inserted 413 count columns
cluster_stat.outliers_cluster_instrucao: inserted outlier rows
cluster_stat.estatisticas_cluster_renda: inserted 413 count columns
cluster_stat.outliers_cluster_renda: inserted outlier rows


In [36]:
table_name = "estatisticas_cluster_idade"
rows = 20
columns_limit = 20

metadata = MetaData(schema=target_schema_name)
table = Table(
    table_name,
    metadata,
    autoload_with=engine,
)

columns = list(table.columns)[:columns_limit]
stmt = select(*columns).limit(rows)

with engine.begin() as connection:
    result = connection.execute(stmt)
    df = pd.DataFrame(result.fetchall(), columns=[column.name for column in columns])

display(df)


,Ano,Cod,coluna_contagem,total_coluna,pop_proj,total_coluna_por_mil,media,mediana,variancia,desvio,q1,q3,limite_outlier_superior
0,2015,0,Total,8700.0,12940354.0,0.672315,0.603793,0.304761,0.837503,0.915152,0.124357,0.652290,1.444189
1,2015,1,Total,10368.0,8280034.0,1.252169,1.219662,0.697407,1.868963,1.367100,0.246682,1.795846,4.119593
2,2015,2,Total,123632.0,122035688.0,1.013081,1.209938,0.733746,2.010010,1.417748,0.340997,1.563518,3.397300
3,2015,3,Total,12845.0,15239137.0,0.842895,0.667772,0.373621,0.690561,0.831000,0.162223,0.845338,1.870011
4,2015,4,Total,5351.0,7496892.0,0.713762,0.673864,0.263473,1.213822,1.101736,0.095983,0.838728,1.952844
5,2015,5,Total,2528.0,3812201.0,0.663134,0.428157,0.197613,0.526762,0.725784,0.089695,0.360955,0.767846
6,2016,0,Total,8160.0,12763756.0,0.639310,0.564343,0.302587,0.572813,0.756844,0.145118,0.666096,1.447563
7,2016,1,Total,9090.0,8313958.0,1.093342,1.168479,0.697647,1.689292,1.299728,0.270849,1.663623,3.752784
8,2016,2,Total,132054.0,123051348.0,1.073162,1.183739,0.771425,1.719978,1.311479,0.333250,1.489008,3.222644
9,2016,3,Total,13154.0,15306465.0,0.859375,0.729928,0.392278,0.958228,0.978891,0.157335,0.834805,1.851010
